In [0]:

# ============================================================
# CELDA 1. DISTRIBUCIONES Y SERIES TEMPORALES DE RETRASOS
# ============================================================

# Preparamos datos agregados para estudiar los retrasos
# de vuelos entre febrero de 2024 y diciembre de 2025.
#
# Utilizamos la tabla Gold operativa.
# Esta etapa es descriptiva: no entrena modelos.

from pyspark.sql import functions as F

TABLA_ORIGEN = "workspace.gold_flights.flights_features"

vuelos = spark.table(TABLA_ORIGEN)

# ------------------------------------------------------------
# 1. CLASIFICAR LOS RETRASOS POR INTERVALOS
# ------------------------------------------------------------

# DepDelay puede ser negativo si un vuelo sale antes
# de su hora programada.

distribucion = (
    vuelos
    .withColumn(
        "delay_category",
        F.when(F.col("dep_delay") < 0, "Salida anticipada")
         .when(F.col("dep_delay") < 15, "0 a 14 minutos")
         .when(F.col("dep_delay") < 30, "15 a 29 minutos")
         .when(F.col("dep_delay") < 60, "30 a 59 minutos")
         .when(F.col("dep_delay") < 120, "60 a 119 minutos")
         .otherwise("120 minutos o más")
    )
    .groupBy("delay_category")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg("dep_delay"), 2).alias("avg_delay_minutes")
    )
)

total = vuelos.count()

distribucion = distribucion.withColumn(
    "percentage",
    F.round(F.col("total_flights") / F.lit(total) * 100, 2)
)

# ------------------------------------------------------------
# 2. SERIE TEMPORAL DIARIA
# ------------------------------------------------------------

# Calculamos el retraso promedio, su variabilidad
# y el porcentaje de vuelos retrasados por fecha.

serie_diaria = (
    vuelos
    .groupBy("flight_date")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg("dep_delay"), 2).alias("avg_delay_minutes"),
        F.round(F.stddev_samp("dep_delay"), 2).alias("std_delay_minutes"),
        F.sum("delayed_15").alias("delayed_flights")
    )
    .withColumn(
        "delay_rate",
        F.round(
            F.col("delayed_flights") / F.col("total_flights") * 100,
            2
        )
    )
    .orderBy("flight_date")
)

# ------------------------------------------------------------
# 3. COMPORTAMIENTO POR HORA PROGRAMADA
# ------------------------------------------------------------

serie_horaria = (
    vuelos
    .groupBy("departure_hour")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg("dep_delay"), 2).alias("avg_delay_minutes"),
        F.round(
            F.avg(F.col("delayed_15")) * 100,
            2
        ).alias("delay_rate")
    )
    .orderBy("departure_hour")
)

# ------------------------------------------------------------
# 4. MOSTRAR RESULTADOS
# ------------------------------------------------------------

print("=" * 55)
print("GOLD: DISTRIBUCIONES Y SERIES TEMPORALES")
print("=" * 55)
print("Total de vuelos analizados:", total)

print("\nDISTRIBUCIÓN DE RETRASOS")
display(distribucion)

print("\nSERIE TEMPORAL DIARIA")
display(serie_diaria.limit(10))

print("\nCOMPORTAMIENTO POR HORA")
display(serie_horaria)

print("\nPREPARACIÓN ESTADÍSTICA COMPLETADA.")


In [0]:

# ============================================================
# CELDA 2. GUARDAR DISTRIBUCIONES Y SERIES TEMPORALES
# ============================================================

# Guardamos tres tablas Gold para los análisis
# estadísticos y las visualizaciones del proyecto.

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.gold_flights
""")

# 1. Distribución de retrasos por intervalos
(
    distribucion.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold_flights.delay_distribution"
    )
)

print("Distribución de retrasos guardada.")

# 2. Serie temporal diaria
(
    serie_diaria.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold_flights.delay_daily_timeseries"
    )
)

print("Serie temporal diaria guardada.")

# 3. Comportamiento por hora programada
(
    serie_horaria.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold_flights.delay_hourly_summary"
    )
)

print("Comportamiento por hora guardado.")

# Verificación final
print("=" * 55)
print("TABLAS GOLD ESTADÍSTICAS CREADAS CORRECTAMENTE")
print("=" * 55)

for nombre in [
    "delay_distribution",
    "delay_daily_timeseries",
    "delay_hourly_summary"
]:
    cantidad = spark.table(
        f"workspace.gold_flights.{nombre}"
    ).count()

    print(f"{nombre}: {cantidad} registros")
